In [65]:
import pandas as pd
#reading the original dataset
df = pd.read_csv("../data/Housing.csv")
#diving the original dataset in 2 subsets, -_X and -_y, which represent the imput and target sets of the dataset 
df_X = df.drop(columns=["price"])
df_y = df["price"]

In [66]:
from sklearn import set_config
#I configured the transformer output to be Panda's Dataframes (made my life a little easier :)  )
set_config(transform_output="pandas")

In [67]:
#the further transforming and pipelining of the dataset requires some clarifications on which
#    columns are numeric and which are object/string ones
#this is easily done with the select_dtypes method, which returns a dataframe with the columns of the specified type
#nums_cols - the numeric columns. In the original dataset they happen to be all int64, so I sticked with blunt assignment
num_cols = df_X.select_dtypes(include=["int64"]).columns.tolist()
#cat_cols - initially string columns that represent categories
cat_cols = df_X.select_dtypes(include=["str"]).columns.tolist()

In [68]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
#now the magic of Pipelines, Imputers, Scalers and Column Transformers happen below...
#transformer for the numeric columns using the standard scaling
#standard scaling is less prone to outliers and exceptions in the dataset, but is less 0-to-1 scaled as well...
stand_num_pipeline = Pipeline([
    ("std_scaler", StandardScaler())
])
#transformer for the numeric columns using the mini-max scaling
#min-max scaling provides normalized values for the column entries, but it's voulnerable to bad data and 
#    evil, sneaky cockroaches that short-circuited registers with their crusty legs...
min_max_num_pipeline = Pipeline([
    ("min_max_scaler", MinMaxScaler())
])

In [69]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
#next come the encoders and transformers
#in order for the models to make use of the string columns, they must be encoded in a numeric way
#2 most popular approaches are:
#    ordinal encoding - replacing each unique string/object with an integer - easy, compact, but relationship agnostic approach
#    one hot encoding - as the name suggest, this encoding tries to find some hotty stringy ones....
#    one hoy encoding - strings are replaced with tuples that contain 0s and 1s, where 1 implies which unique string values is used
#next come the transformers - they are DMs to Optimus Prime with a request to make proper columns....
#transformers - special objects that apply some transormations over some columns according to the list of tuples given in the constructor
#    input tuple includes - the "label" of the transformation, operation to be done over the columns, and the solumns to be transformed

#below I wrote 2 ones, one with the standard scaling and ordinal encoding, and its counterpart with min-max scaling instead of standard one
std_ord_pl = ColumnTransformer([
    ("num", stand_num_pipeline, num_cols),
    ("cat", OrdinalEncoder(), cat_cols)
])

mm_ord_pl = ColumnTransformer([
    ("num", min_max_num_pipeline, num_cols),
    ("cat", OrdinalEncoder(), cat_cols)
])
#we apply the transformations to the INPUT dataset we separated earlier
#    this is a key moment, the target columns mustn't be scaled and transformed, UNLESS there are specific conditions that require them
#here are the 2 newly datasets made "ca la oameni", which will be further used for training the LM models
std_ord_df = std_ord_pl.fit_transform(df_X)
mm_ord_df = mm_ord_pl.fit_transform(df_X)
#adding the target column of course
std_ord_df["price"] = df_y
mm_ord_df["price"] = df_y
#further storing them in the data/ directory for later use
std_ord_df.to_csv("../data/std_ord_df.csv")
mm_ord_df.to_csv("../data/mm_ord_df.csv")

In [70]:
#unfortunatelly, transforming the columns changed their names as well
#    even thou I like the intriguing "num__" and "cat__" prefixes to the names, that seems as an upcoming problem
std_ord_df.head(10)

,num__area,num__bedrooms,num__bathrooms,num__stories,num__parking,cat__mainroad,cat__guestroom,cat__basement,cat__hotwaterheating,cat__airconditioning,cat__prefarea,cat__furnishingstatus,price
0,1.046726,1.403419,1.421812,1.378217,1.517692,1.0,0.0,0.0,0.0,1.0,1.0,0.0,13300000
1,1.757010,1.403419,5.405809,2.532024,2.679409,1.0,0.0,0.0,0.0,1.0,0.0,0.0,12250000
2,2.218232,0.047278,1.421812,0.224410,1.517692,1.0,0.0,1.0,0.0,0.0,1.0,1.0,12250000
3,1.083624,1.403419,1.421812,0.224410,2.679409,1.0,0.0,1.0,0.0,1.0,1.0,0.0,12215000
4,1.046726,1.403419,-0.570187,0.224410,1.517692,1.0,1.0,1.0,0.0,1.0,0.0,0.0,11410000
5,1.083624,0.047278,3.413810,-0.929397,1.517692,1.0,0.0,1.0,0.0,1.0,1.0,1.0,10850000
6,1.581745,1.403419,3.413810,2.532024,1.517692,1.0,0.0,0.0,0.0,1.0,1.0,1.0,10150000
7,5.096263,2.759560,3.413810,0.224410,-0.805741,1.0,0.0,0.0,0.0,0.0,0.0,2.0,10150000
8,1.360358,1.403419,-0.570187,0.224410,1.517692,1.0,1.0,1.0,0.0,1.0,1.0,0.0,9870000
9,0.276484,0.047278,1.421812,2.532024,0.355976,1.0,1.0,0.0,0.0,1.0,1.0,2.0,9800000


In [71]:
std_ord_df.dtypes

num__area                float64
num__bedrooms            float64
num__bathrooms           float64
num__stories             float64
num__parking             float64
cat__mainroad            float64
cat__guestroom           float64
cat__basement            float64
cat__hotwaterheating     float64
cat__airconditioning     float64
cat__prefarea            float64
cat__furnishingstatus    float64
price                      int64
dtype: object

In [72]:
mm_ord_df.head(10)

,num__area,num__bedrooms,num__bathrooms,num__stories,num__parking,cat__mainroad,cat__guestroom,cat__basement,cat__hotwaterheating,cat__airconditioning,cat__prefarea,cat__furnishingstatus,price
0,0.396564,0.6,0.333333,0.666667,0.666667,1.0,0.0,0.0,0.0,1.0,1.0,0.0,13300000
1,0.502405,0.6,1.000000,1.000000,1.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,12250000
2,0.571134,0.4,0.333333,0.333333,0.666667,1.0,0.0,1.0,0.0,0.0,1.0,1.0,12250000
3,0.402062,0.6,0.333333,0.333333,1.000000,1.0,0.0,1.0,0.0,1.0,1.0,0.0,12215000
4,0.396564,0.6,0.000000,0.333333,0.666667,1.0,1.0,1.0,0.0,1.0,0.0,0.0,11410000
5,0.402062,0.4,0.666667,0.000000,0.666667,1.0,0.0,1.0,0.0,1.0,1.0,1.0,10850000
6,0.476289,0.6,0.666667,1.000000,0.666667,1.0,0.0,0.0,0.0,1.0,1.0,1.0,10150000
7,1.000000,0.8,0.666667,0.333333,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,2.0,10150000
8,0.443299,0.6,0.000000,0.333333,0.666667,1.0,1.0,1.0,0.0,1.0,1.0,0.0,9870000
9,0.281787,0.4,0.333333,1.000000,0.333333,1.0,1.0,0.0,0.0,1.0,1.0,2.0,9800000
